## 准备数据

In [13]:
import os
import numpy as np
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers, optimizers, datasets

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

# 定义数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),  # 将图像转换为张量，并自动归一化到 [0, 1]
    transforms.Normalize((0.5,), (0.5,))  # 将数据归一化到 [-1, 1]
])

# 加载训练集和测试集
train_dataset = datasets.MNIST(
    root='./data',          # 数据保存路径
    train=True,             # 加载训练集
    download=True,          # 如果数据不存在，自动下载
    transform=transform     # 应用预处理
)
test_dataset = datasets.MNIST(
    root='./data',          # 数据保存路径
    train=False,            # 加载测试集
    download=True,          # 如果数据不存在，自动下载
    transform=transform     # 应用预处理
)

# 将整个数据集转换为单个张量
train_images = torch.stack([img for img, _ in train_dataset])  # [60000, 1, 28, 28]
train_labels = torch.tensor([label for _, label in train_dataset])  # [60000]
print(type(train_images))
print(type(train_labels))

test_images = torch.stack([img for img, _ in test_dataset])  # [10000, 1, 28, 28]
test_labels = torch.tensor([label for _, label in test_dataset])  # [10000]

<class 'torch.Tensor'>
<class 'torch.Tensor'>


## 建立模型

In [6]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


In [8]:
class myModel(nn.Module):
    def __init__(self):
        super(myModel, self).__init__()
        # 定义模型的参数
        self.fc1 = nn.Linear(28 * 28, 100)  # 输入层到隐藏层
        self.fc2 = nn.Linear(100, 10)       # 隐藏层到输出层
        
    def __call__(self, x):
        x = x.view(-1, 28*28)  # 将输入展平
        self.h1_relu = torch.relu(self.fc1(x))
        self.h2_relu = self.fc2(self.h1_relu)
        
        return self.h2_relu
        

model = myModel()

optimizer = optim.SGD(model.parameters(), lr=0.01)  # 使用梯度下降

## 计算 loss

In [19]:
criterion = nn.CrossEntropyLoss()
def compute_loss(outputs, labels):
    return criterion(outputs, labels)

def compute_accuracy(logits, labels):
    # 获取预测结果（取 logits 中最大值的索引）
    predictions = torch.argmax(logits, dim=1)
    # 计算预测正确的数量
    correct = (predictions == labels).sum().item()
    # 计算精确度
    accuracy = correct / labels.size(0)
    return accuracy

def train_one_step(model, optimizer, x, y):
    model.train()  # 将模型设置为训练模式
    # 前向传播
    outputs = model(x)  # 使用整个训练集
    loss = compute_loss(outputs, y)
    accuracy = compute_accuracy(outputs, y)
    # 反向传播和优化
    optimizer.zero_grad()  # 清零梯度
    loss.backward()        # 计算梯度
    optimizer.step()       # 更新参数
    return loss, accuracy

def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [24]:
# train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, train_images, train_labels)
    print('epoch', epoch, ': loss', loss.item(), '; accuracy', accuracy)
loss, accuracy = test(model, test_images, test_labels)

print('test loss', loss.item(), '; accuracy', accuracy)

epoch 0 : loss 1.8653650283813477 ; accuracy 0.6432833333333333
epoch 1 : loss 1.856506586074829 ; accuracy 0.6463333333333333
epoch 2 : loss 1.8476186990737915 ; accuracy 0.6493666666666666
epoch 3 : loss 1.8387032747268677 ; accuracy 0.6521833333333333
epoch 4 : loss 1.829771876335144 ; accuracy 0.6549166666666667
epoch 5 : loss 1.8208255767822266 ; accuracy 0.6579
epoch 6 : loss 1.8118685483932495 ; accuracy 0.6607166666666666
epoch 7 : loss 1.8029125928878784 ; accuracy 0.6631666666666667
epoch 8 : loss 1.7939553260803223 ; accuracy 0.6657
epoch 9 : loss 1.785004734992981 ; accuracy 0.6680333333333334
epoch 10 : loss 1.7760614156723022 ; accuracy 0.6702666666666667
epoch 11 : loss 1.7671282291412354 ; accuracy 0.6721166666666667
epoch 12 : loss 1.7582076787948608 ; accuracy 0.6743666666666667
epoch 13 : loss 1.7492985725402832 ; accuracy 0.6767833333333333
epoch 14 : loss 1.7403984069824219 ; accuracy 0.6787833333333333
epoch 15 : loss 1.7315115928649902 ; accuracy 0.6803
epoch 16 